In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader

DATA_PATH = "documents"

# Check folder exists
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Folder not found: {DATA_PATH}")

print("✅ Documents folder found")

# Load all PDF files
documents = []

for file in os.listdir(DATA_PATH):
    if file.lower().endswith(".pdf"):
        pdf_path = os.path.join(DATA_PATH, file)

        print(f"📄 Loading: {file}")

        loader = PyPDFLoader(pdf_path)
        documents.extend(loader.load())

print(f"\n✅ Total Pages Loaded: {len(documents)}")

✅ Documents folder found
📄 Loading: medical_policy.pdf

✅ Total Pages Loaded: 9


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

texts = text_splitter.split_documents(documents)

print(f"✅ Total Chunks Created: {len(texts)}")

✅ Total Chunks Created: 26


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding Model Loaded")

✅ Embedding Model Loaded


In [14]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=texts,
    embedding=embedding_model
)

print("✅ FAISS Vector Store Created")
print("Total Chunks Indexed:", vectorstore.index.ntotal)

✅ FAISS Vector Store Created
Total Chunks Indexed: 26


In [15]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("✅ Retriever Created")

✅ Retriever Created


In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("✅ FLAN-T5 Model Loaded")

✅ FLAN-T5 Model Loaded


In [17]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

pipe = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0,
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ FLAN-T5 Pipeline Ready")

Device set to use mps:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ FLAN-T5 Pipeline Ready


In [18]:
from langchain_core.prompts import PromptTemplate

template = """
You are a helpful AI assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

print("✅ Prompt Template Ready")

✅ Prompt Template Ready


In [24]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

def ask_question(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt_text = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt_text)

    return response

In [25]:
question = "What is covered under the medical policy?"

answer = ask_question(question)

print(answer)

Cosmetic surgery unless medically necessary  Weight reduction treatments  Fertility treatment  Treatment for infertility unless covered separately  Self-inflicted injuries  Suicide attempts  Alcohol or drug-related hospitalization  Non-medical expenses  Vitamins and supplements without medical necessity  Hearing aids  Spectacles and contact lenses   Doctor consultations  Diagnostic tests  Prescribed medicines Subject to policy terms. Specialized Care Plans Elder Care Plan Designed to support senior family members. Coverage may include:  Preventive healthcare  Senior wellness services Chronic Condition Management Plan Coverage may include:  Diabetes management  Hypertension management  Long-term disease support supplements without medical necessity  Hearing aids  Spectacles and contact lenses  Dental treatment unless hospitalization is required  Non-prescribed alternative treatments  Expenses related to terrorism as per policy exclusions


In [ ]:
while True:
    q = input("Ask (type 'exit' to quit): ")

    if q.lower() == "exit":
        break

    print("\nSearching and fetching answer...")

    answer = ask_question(q)

    print("\nAnswer:")
    print(answer)
    print("-" * 60)


Ask (type 'exit' to quit):  What is covered under the medical policy?



Searching and fetching answer...

Answer:
Cosmetic surgery unless medically necessary  Weight reduction treatments  Fertility treatment  Treatment for infertility unless covered separately  Self-inflicted injuries  Suicide attempts  Alcohol or drug-related hospitalization  Non-medical expenses  Vitamins and supplements without medical necessity  Hearing aids  Spectacles and contact lenses   Doctor consultations  Diagnostic tests  Prescribed medicines Subject to policy terms. Specialized Care Plans Elder Care Plan Designed to support senior family members. Coverage may include:  Preventive healthcare  Senior wellness services Chronic Condition Management Plan Coverage may include:  Diabetes management  Hypertension management  Long-term disease support supplements without medical necessity  Hearing aids  Spectacles and contact lenses  Dental treatment unless hospitalization is required  Non-prescribed alternative treatments  Expenses related to terrorism as per policy exclusions
------

Ask (type 'exit' to quit):  who to contact for more details



Searching and fetching answer...

Answer:
Cashless Hospitalization Support Rahul Mehta / Sneha Kulkarni +91 90000 12345 / +91 90000 23456 cashless.support@healthassist-example.com Claim Reimbursement Support Priya Sharma / Karan Joshi +91 90123 45678 / +91 90234 56789 Priya Sharma / Karan Joshi +91 90123 45678 / +91 90234 56789 claims.support@insurancebroker-example.com
------------------------------------------------------------


Ask (type 'exit' to quit):  how to add new dependent now


...


Searching and fetching answer...

Answer:
Optional Add-on Coverage
------------------------------------------------------------


Ask (type 'exit' to quit):  whats the Policy Coverage Period?



Searching and fetching answer...

Answer:
Policy Start Date: 01 July 2026 Policy End Date: 30 June 2027 All eligible employees and enrolled dependents will be covered during the policy period, subject to the terms and conditions of the insurance policy.
------------------------------------------------------------


Ask (type 'exit' to quit):  whats the Eligibility Criteria?



Searching and fetching answer...

Answer:
The policy covers: Employees  Full-time employees of the organization  Eligible employees as per company policy guidelines Eligible Dependents Employees may include:  Spouse / Partner  Children  Parents After submission:  Confirmation email will be generated.  Selected benefits will remain active for the policy year. Default Enrollment Employees who do not complete enrollment within the given timeline will be automatically enrolled into the default plan. Example Default Coverage:  Employee  Spouse  Two children  Two parents/parents-in-law Coverage:  Sum Insured: INR 5,00,000  Parent sub-limit: INR 3,00,000 Covered Member Intern Coverage Type Self Only Sum Insured INR 3,00,000 Enrollment Automatic Coverage Duration Internship Period Insurance Support Contacts Support Area Contact Person Phone Email Cashless Hospitalization Support Rahul Mehta / Sneha Kulkarni +91 90000 12345 / +91 90000 23456 cashless.support@healthassist-example.com Claim Re
-